# Lumen Travel · Ollama + Pinggy

Chọn **Runtime → Change runtime type → T4 GPU** rồi chạy các ô theo thứ tự.
Setup chạy nền; kernel vẫn nhận lệnh. Ô cuối hiển thị tiến trình và hai giá trị
cần chép vào `.env`. Không lưu output chứa token vào GitHub.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
repo=Path("/content/lumen-travel-assistant")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/vo-m-tri1412/lumen-travel-assistant.git",str(repo)],check=True,timeout=90)
else:
    subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True,timeout=90)
log=open("/tmp/lumen-launch.log","a")
job=subprocess.Popen([sys.executable,"-m","server.start_server"],cwd=repo,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print("Setup chạy nền. PID:",job.pid)


In [ ]:
status_path=Path("/tmp/lumen-server.json")
last_stage=None
deadline=time.monotonic()+900
while time.monotonic()<deadline:
    if status_path.exists():
        status=json.loads(status_path.read_text())
        if status.get("stage")!=last_stage:
            print(status.get("stage"),flush=True)
            last_stage=status.get("stage")
        if status.get("ready"):
            print("OLLAMA_BASE_URL="+status["url"])
            print("OLLAMA_API_TOKEN="+status["api_token"])
            break
        if status.get("stage")=="error":
            print(Path("/tmp/lumen-setup.log").read_text()[-4000:])
            raise RuntimeError(status.get("error","Setup failed"))
    time.sleep(2)
else:
    raise RuntimeError("Hết thời gian chờ. Xem /tmp/lumen-setup.log.")
